## optuna 설치

In [ ]:
!pip install optuna==4.6.0 optuna-integration==4.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 6.4 MB/s eta 0:00:00


### Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 평가지표 함수 선언

In [ ]:
import sys
import os

sys.path.insert(0, '/content/drive/MyDrive/pill_detection_project')

from src.evaluation import (
    evaluate_all,
    init_history,
    update_history,
    save_history,
    load_history,
    plot_training_history,
    plot_compare_histories,
    convert_yolo_results,
    convert_torchvision_outputs,
)

## 전처리 코드

| Step | 이름 | 설명 |
|------|------|------|
| Step 1 | Stratified Split | 원본 JSON → train_raw.json / val.json (9:1 분할) |
| Step 1-B | 소수 클래스 추출 | 50개 미만 클래스 객체 잘라서 crops_minority/ 저장 |
| Step 2 | Copy-Paste 증강 | 소수 클래스를 다른 이미지에 합성 (4,095개 → 6,199개) |
| Step 3 | Letterbox 변환 | 모든 이미지를 800×800으로 통일 |
| Step 4 | CLAHE 대비 강화 | 알약 각인 잘 보이게 대비 강화 |
| Step 5 | YOLO 라벨 변환 | YOLO용 .txt 포맷 변환 + data.yaml 생성 (YOLO팀용) |

In [ ]:
# 전처리 코드 전처리가 완료된 경우 실행 불필요
# !python /content/drive/MyDrive/pill_detection_project/run_preprocessing.py

## DataLoader

In [ ]:
import torch
import torchvision
import torch.optim as optim
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from src.preprocessing.dataset import get_loaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'
train_loader, val_loader, orig2model, num_classes, val_json = get_loaders(base_dir=BASE_DIR)

TEST_JSON_PATH = BASE_DIR + '/merged_annotations_test_final.json'

✅ 고유 클래스 수  : 73종
✅ num_classes     : 74  ← 모델 정의 시 사용
✅ Train: 1800장 / 6189개
✅ Val  : 139장 / 431개


### 모델 정의

#### ✅ 변경사항
- **[개선]** Backbone을 `resnet50` → `mobilenet_v3_large` 로 교체
  - 학습 속도 단축 목적
  - MobileNetV3는 rpn_anchor_generator 별도 설정 불가 (기본 Anchor Size 사용)


### 모델 정의 및 학습 루프

#### ✅ 변경사항 요약
| 항목 | 베이스라인 (v1) | 개선 (v2 Final) |
|------|----------------|----------|
| Scheduler | StepLR (step=3, γ=0.1) | **CosineAnnealingLR** |
| Early Stopping | ❌ 없음 | **✅ patience=5, mAP@50 기준** |
| Gradient Clipping | ❌ 없음 | **✅ max_norm=5.0** |
| Backbone | ResNet50 FPN | **MobileNetV3 Large FPN** |
| lr | 1e-4 | **0.000129 (Optuna 최적값)** |
| weight_decay | ❌ | **0.000187 (Optuna 최적값)** |
| eta_min | - | **0.000058 (Optuna 최적값)** |
| 랜덤 시드 | ❌ | **✅ seed=42 고정** |


In [ ]:
############################################################
#  랜덤 시드 고정
############################################################

import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


############################################################
#  모델 정의 (Faster R-CNN + MobileNetV3 Large FPN)
#  ⚠️ MobileNetV3는 rpn_anchor_generator 별도 설정 불가
#  → 기본 Anchor Size 사용
############################################################

def build_model_fasterrcnn_v2(num_classes):
    model = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(
        weights="DEFAULT"
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    model.to(DEVICE)
    return model


############################################################
# 모델 빌드
############################################################

model = build_model_fasterrcnn_v2(num_classes)


############################################################
#  학습 루프 v2 Final
############################################################

def train_fasterrcnn_v2(model, train_loader, val_loader, num_epochs=10):
    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.000129,
        weight_decay=0.000187
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=0.000058
    )

    patience = 5
    best_map = 0.0
    early_stop_counter = 0
    best_model_state = None

    history = init_history()

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            train_loss_sum += losses.item()

        avg_train_loss = train_loss_sum / max(1, len(train_loader))

        model.train()
        val_loss_sum = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(DEVICE) for img in images]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                val_loss_sum += losses.item()

        avg_val_loss = val_loss_sum / max(1, len(val_loader))

        all_outputs = []
        all_image_ids = []
        model.eval()
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(DEVICE) for img in images]
                outputs = model(images)
                batch_image_ids = [t["image_id"].item() for t in targets]
                all_outputs.extend(outputs)
                all_image_ids.extend(batch_image_ids)

        val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

        metrics = evaluate_all(
            gt_json_path=val_json,
            predictions=val_predictions,
            conf_threshold=0.2,
            pr_iou_threshold=0.5,
            temp_json_path=f"faster_rcnn_v2_final_epoch_{epoch}.json",
            model2orig={v: k for k, v in orig2model.items()}
        )

        current_map = metrics.get("mAP@50", 0.0)

        update_history(history, epoch=epoch, train_loss=avg_train_loss,
                       val_loss=avg_val_loss, metrics=metrics)

        current_lr = scheduler.get_last_lr()[0]
        print(f"[Epoch {epoch}/{num_epochs}] "
              f"train_loss: {avg_train_loss:.4f} | "
              f"val_loss: {avg_val_loss:.4f} | "
              f"mAP@50: {current_map:.4f} | "
              f"lr: {current_lr:.6f}")

        scheduler.step()

        if current_map > best_map:
            best_map = current_map
            early_stop_counter = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f"  ✅ mAP@50 개선! best mAP@50: {best_map:.4f}")
        else:
            early_stop_counter += 1
            print(f"  ⚠️  mAP@50 개선 없음 ({early_stop_counter}/{patience})")
            if early_stop_counter >= patience:
                print(f"\n🛑 Early Stopping! {patience} epoch 동안 mAP@50 개선 없어 학습 종료")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✅ Best 모델 복원 완료 (mAP@50: {best_map:.4f})")

    torch.save(model.state_dict(), "fasterrcnn_v2_final.pth")
    save_history(history, "history_faster_rcnn_v2_final.json")
    plot_training_history(history, title_prefix="Faster R-CNN v2 Final")
    print("모델 저장 완료")
    return model


############################################################
# 함수 호출 (주석처리 - 학습 없이 가중치 불러오기)
############################################################

# model = train_fasterrcnn_v2(model, train_loader, val_loader, num_epochs=10)


############################################################
# 드라이브에서 가중치 불러오기
############################################################

model.load_state_dict(torch.load("/content/drive/MyDrive/fasterrcnn_v2_final.pth", map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("✅ 가중치 불러오기 완료!")

Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth


100%|██████████| 74.2M/74.2M [00:00<00:00, 147MB/s]


✅ 가중치 불러오기 완료!


## Optuna 하이퍼파라미터 튜닝

| 항목 | 설정 |
|------|------|
| 탐색 파라미터 | lr, weight_decay, eta_min |
| 탐색 범위 (lr) | 1e-6 ~ 1e-2 |
| 탐색 범위 (weight_decay) | 1e-6 ~ 1e-1 |
| 탐색 범위 (eta_min) | 1e-7 ~ 1e-4 |
| Trial 수 | 20번 |
| 목표 | mAP@50 최대화 |
| Pruner | MedianPruner (성능 낮은 trial 자동 제거) |

### Optuna 결과
| 항목 | trial 10 | trial 20 (최종) |
|------|----------|----------------|
| lr | 0.000248 | **0.000129** |
| weight_decay | 0.000006 | **0.000187** |
| eta_min | 0.0000003 | **0.000058** |
| Best mAP@50 | 0.9166 | **0.9190** |

### 최종 학습 결과 (Optuna 최적값 적용)
| 항목 | 결과 |
|------|------|
| mAP@50 | **0.9202** |
| 학습 시간 | 약 7~11분 (epoch 10) |
| Early Stopping | 5 epoch 기준 자동 종료 |

> Optuna trial 20 최적값 적용 후 실제 학습에서 **mAP@50 0.9202** 달성
> 베이스라인(0.893) 대비 **+0.027 향상**


## 추론

### 추론 방식
| 항목 | 설정 |
|------|------|
| 이미지 전처리 | **Letterbox 변환 (800×800)** |
| 박스 좌표 | **원본 이미지 좌표로 복원** |
| score_threshold | 0.3 |
| TOP_K per image | 4 |

> test 이미지가 976×1280 원본 크기이므로 학습 시 사용한 800×800 letterbox로 변환 후 추론
> 예측된 박스 좌표를 원본 이미지 크기로 복원하여 정확한 좌표 사용


In [ ]:
############################################################
# 추론 - test_images에 대해 예측 → submission.csv 생성
############################################################

import os
import pandas as pd
from PIL import Image
import torchvision.transforms as T
import shutil

# letterbox 변환 함수
def letterbox_pil_for_inference(image, target_size=800, fill=(114, 114, 114)):
    orig_w, orig_h = image.size
    scale = target_size / max(orig_w, orig_h)
    new_w = max(1, int(orig_w * scale))
    new_h = max(1, int(orig_h * scale))

    resized = image.resize((new_w, new_h), Image.BILINEAR)
    padded = Image.new("RGB", (target_size, target_size), fill)

    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2
    padded.paste(resized, (pad_left, pad_top))

    meta = {
        "orig_w": orig_w,
        "orig_h": orig_h,
        "scale": scale,
        "pad_left": pad_left,
        "pad_top": pad_top,
    }
    return padded, meta


# 박스 좌표 원본으로 복원
def restore_boxes_to_original_xyxy(boxes, meta):
    if boxes.numel() == 0:
        return boxes

    restored = boxes.clone().float()
    restored[:, [0, 2]] = (restored[:, [0, 2]] - meta["pad_left"]) / meta["scale"]
    restored[:, [1, 3]] = (restored[:, [1, 3]] - meta["pad_top"]) / meta["scale"]
    restored[:, [0, 2]] = restored[:, [0, 2]].clamp(0, meta["orig_w"])
    restored[:, [1, 3]] = restored[:, [1, 3]].clamp(0, meta["orig_h"])
    return restored


TEST_IMG_DIR = BASE_DIR + '/test_images'
SCORE_THRESHOLD = 0.3
TOP_K_PER_IMAGE = 4

# model2orig 정의
model2orig = {v: k for k, v in orig2model.items()}

test_files = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")])
print(f"총 test 이미지 수: {len(test_files)}")

model.eval()
rows = []

with torch.no_grad():
    for f in test_files:
        img_path = os.path.join(TEST_IMG_DIR, f)
        image = Image.open(img_path).convert("RGB")
        image_id = os.path.splitext(f)[0]

        # letterbox 변환
        letterboxed_image, letterbox_meta = letterbox_pil_for_inference(image)

        img_tensor = T.ToTensor()(letterboxed_image).to(DEVICE)
        outputs = model([img_tensor])[0]

        boxes = outputs["boxes"].detach().cpu()
        labels = outputs["labels"].detach().cpu()
        scores = outputs["scores"].detach().cpu()

        # 박스 좌표 원본으로 복원
        boxes = restore_boxes_to_original_xyxy(boxes, letterbox_meta)

        keep = scores >= SCORE_THRESHOLD
        boxes = boxes[keep]
        labels = labels[keep]
        scores = scores[keep]

        # 상위 K개 선택
        if scores.numel() > TOP_K_PER_IMAGE:
            scores, topk_idx = scores.topk(TOP_K_PER_IMAGE)
            boxes = boxes[topk_idx]
            labels = labels[topk_idx]

        for box, lab, sc in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.tolist()
            orig_cat = model2orig[int(lab.item())]
            rows.append({
                "image_id": image_id,
                "category_id": int(orig_cat + 1),  # 원본 약품코드 + 1
                "bbox_x": float(x1),
                "bbox_y": float(y1),
                "bbox_w": float(x2 - x1),
                "bbox_h": float(y2 - y1),
                "score": float(sc.item()),
            })

df_sub = pd.DataFrame(rows, columns=[
    "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])

df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

output_path = "fasterrcnn_v2_submission.csv"
df_sub.to_csv(output_path, index=False)
print(f"✅ 생성 완료: {output_path}")
print(f"📊 총 예측 객체 수: {len(df_sub)}")
print(df_sub.head())

shutil.copy(output_path, f"/content/drive/MyDrive/{output_path}")
print("✅ 드라이브 저장 완료!")

총 test 이미지 수: 843
✅ 생성 완료: fasterrcnn_v2_submission.csv
📊 총 예측 객체 수: 3196
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1           24  562.799011   98.206154  400.741821   
1              2        1           45  168.994537  772.360291  168.316895   
2              3        1            1  164.264603  250.676910  200.540512   
3              4        1           52  599.871460  662.623047  238.952148   
4              5       10           56  108.507812  248.538132  404.107239   

       bbox_h     score  
0  364.687981  0.929670  
1  258.360657  0.902687  
2  122.974762  0.833534  
3  471.800781  0.823247  
4  218.742050  0.964107  
✅ 드라이브 저장 완료!


In [ ]:
# ############################################################
# #  Optuna 하이퍼파라미터 튜닝
# #  탐색 파라미터: lr, weight_decay, eta_min
# #  목표: mAP@50 최대화
# #  trial: 20번 (10번 → 20번으로 증가)
# ############################################################

# import optuna
# import random
# import numpy as np

# def set_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

# def objective(trial):
#     set_seed(42)

#     # 매 trial마다 모델 새로 빌드
#     model = build_model_fasterrcnn_v2(num_classes)

#     # 탐색할 하이퍼파라미터 범위 설정
#     lr = trial.suggest_float('lr', 1e-6, 1e-2, log=True)
#     weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-1, log=True)
#     eta_min = trial.suggest_float('eta_min', 1e-7, 1e-4, log=True)

#     optimizer = optim.AdamW(
#         [p for p in model.parameters() if p.requires_grad],
#         lr=lr,
#         weight_decay=weight_decay
#     )

#     scheduler = optim.lr_scheduler.CosineAnnealingLR(
#         optimizer,
#         T_max=10,
#         eta_min=eta_min
#     )

#     # Early Stopping 설정
#     patience = 5
#     best_map = 0.0
#     early_stop_counter = 0

#     for epoch in range(1, 11):
#         # -------------------------------------------------------
#         # 1) Train
#         # -------------------------------------------------------
#         model.train()
#         train_loss_sum = 0.0
#         for images, targets in train_loader:
#             images = [img.to(DEVICE) for img in images]
#             targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

#             loss_dict = model(images, targets)
#             losses = sum(loss for loss in loss_dict.values())

#             optimizer.zero_grad()
#             losses.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
#             optimizer.step()
#             train_loss_sum += losses.item()

#         # -------------------------------------------------------
#         # 2) 검증셋 예측
#         # -------------------------------------------------------
#         all_outputs = []
#         all_image_ids = []
#         model.eval()
#         with torch.no_grad():
#             for images, targets in val_loader:
#                 images = [img.to(DEVICE) for img in images]
#                 outputs = model(images)
#                 batch_image_ids = [t["image_id"].item() for t in targets]
#                 all_outputs.extend(outputs)
#                 all_image_ids.extend(batch_image_ids)

#         val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

#         # -------------------------------------------------------
#         # 3) 평가
#         # -------------------------------------------------------
#         metrics = evaluate_all(
#             gt_json_path=val_json,
#             predictions=val_predictions,
#             conf_threshold=0.2,
#             pr_iou_threshold=0.5,
#             temp_json_path=f"optuna_trial_{trial.number}_epoch_{epoch}.json",
#             model2orig={v: k for k, v in orig2model.items()}
#         )

#         current_map = metrics.get('mAP@50', 0.0)

#         print(f"  Trial {trial.number} | Epoch {epoch} | mAP@50: {current_map:.4f} | lr: {lr:.6f}")

#         # Optuna 중간 보고 (pruning 용)
#         trial.report(current_map, epoch)
#         if trial.should_prune():
#             raise optuna.exceptions.TrialPruned()

#         # Early Stopping
#         if current_map > best_map:
#             best_map = current_map
#             early_stop_counter = 0
#         else:
#             early_stop_counter += 1
#             if early_stop_counter >= patience:
#                 print(f"  🛑 Early Stopping! (Trial {trial.number})")
#                 break

#         scheduler.step()

#     return best_map


# ############################################################
# # Optuna 실행
# ############################################################

# sampler = optuna.samplers.TPESampler(seed=42)
# pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=3)

# study = optuna.create_study(
#     direction='maximize',
#     sampler=sampler,
#     pruner=pruner
# )

# study.optimize(objective, n_trials=20)

# # 결과 출력
# print("\n" + "="*50)
# print("✅ Optuna 탐색 완료!")
# print(f"  Best mAP@50: {study.best_value:.4f}")
# print(f"  Best params:")
# for key, value in study.best_params.items():
#     print(f"    {key}: {value:.6f}")

## 📊 v1 vs v2 실험 결과 비교

### 모델 변경사항
| 항목 | v1 (베이스라인) | v2 Final (개선) |
|------|----------------|-----------|
| Scheduler | StepLR | CosineAnnealingLR |
| Early Stopping | ❌ | ✅ patience=5, mAP@50 기준 |
| Gradient Clipping | ❌ | ✅ max_norm=5.0 |
| Backbone | ResNet50 FPN | MobileNetV3 Large FPN |
| Weight Decay | ❌ | ✅ 0.000187 (Optuna 최적값) |
| conf_threshold | 0.25 | 0.2 |
| lr | 1e-4 | 0.000129 (Optuna 최적값) |
| 랜덤 시드 | ❌ | ✅ seed=42 고정 |

### 성능 비교 (epoch 10 기준)
| 지표 | v1 (ResNet50) | v2 Final (MobileNetV3) | 변화 |
|------|--------------|----------------------|------|
| mAP@50 | 0.893 | **0.9202** | 📈 +0.027 |
| mAP (0.50:0.95) | 0.746 | 0.679 | 📉 -0.067 |
| AP @ 0.50 | 0.893 | 0.922 | 📈 +0.029 |
| AP @ 0.75 | 0.877 | 0.837 | 📉 -0.040 |
| Recall | 0.989 | 0.988 | ➡️ 유지 |

### 학습 시간 비교
| | v1 | v2 Final |
|--|----|---------|
| 학습 시간 (epoch 10) | 28분 | **약 7~11분** |
| 단축률 | - | **약 60~75% 단축** |

### 학습 곡선 분석
| 지표 | 추세 | 분석 |
|------|------|------|
| Train Loss | 📉 꾸준히 감소 | 정상적인 학습 진행 |
| Val Loss | 📉 안정적으로 수렴 | ✅ 양호 |
| mAP@50 | 📈 0.90 이상 안정적 | ✅ 양호 |
| Precision | 📈 꾸준히 우상향 | ✅ 양호 |
| Recall | ➡️ 0.98 이상 안정적 유지 | ✅ mAP@50 Early Stopping으로 해결 |

### 결론
- mAP@50 기준 v1(0.893) → v2 Final(0.9202)으로 **성능 향상**
- 학습 시간 28분 → 7~11분으로 **대폭 단축**
- Recall 하락 문제 → **mAP@50 Early Stopping으로 해결 완료**
- 과적합 징조 → **Weight Decay + Gradient Clipping으로 해결 완료**
- Optuna trial 20으로 최적 하이퍼파라미터 탐색 완료
- **v2 Final (mAP@50: 0.9202) 최종 확정**

## 깃허브

In [ ]:
!cd /content/drive/MyDrive/pill_detection_project && git add src/models/fasterrcnn/fasterrcnn_v2.ipynb && git commit -m "docs: 마크다운 수정 및 추론 코드 letterbox 적용" && git push origin feat/fasterrcnn-v2-improvements

[feat/fasterrcnn-v2-improvements 5a45d87] docs: 마크다운 수정 및 추론 코드 letterbox 적용
 1 file changed, 1 insertion(+), 890 deletions(-)
 rewrite src/models/fasterrcnn/fasterrcnn_v2.ipynb (99%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 9.36 KiB | 504.00 KiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/wina0901/pill_detection_project.git
   c47a5f4..5a45d87  feat/fasterrcnn-v2-improvements -> feat/fasterrcnn-v2-improvements


In [ ]:
!git config --global user.email "chany1634@gmail.com"
!git config --global user.name "chany1634-kr"